# Intervention Fork, Replay, and Bundle Workflow

This 50-minute-track notebook is intentionally self-contained: it uses a seeded tiny PyTorch model, no downloads, and only public TorchLens APIs. The goal is to show the full workflow shape you would use on a larger model while keeping every cell runnable on CPU.


Fork-replay-bundle is the TorchLens workflow for controlled counterfactuals: capture once, branch the log, apply interventions, propagate downstream, then compare branches with a shared graph-aware object.


In [ ]:
from __future__ import annotations

import math
from typing import Any

import matplotlib.pyplot as plt
import torch
from torch import nn

import torchlens as tl

SEED = 1604
torch.manual_seed(SEED)
torch.set_grad_enabled(False)


class TinyMLP(nn.Module):
    """Small MLP with two intervention-friendly nonlinearities."""

    def __init__(self) -> None:
        """Initialize layers."""
        super().__init__()
        self.in_proj = nn.Linear(8, 8)
        self.mid_proj = nn.Linear(8, 8)
        self.out_proj = nn.Linear(8, 3)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Run the MLP."""
        x = torch.relu(self.in_proj(x))
        x = torch.relu(self.mid_proj(x))
        return self.out_proj(x)

In [ ]:
model = TinyMLP().eval()
x = torch.randn(4, 8)
clean = tl.log_forward_pass(model, x, vis_opt="none", intervention_ready=True, name="clean")
relu_labels = [site.layer_label for site in clean.find_sites(tl.func("relu"))]
print(relu_labels)

In [ ]:
zero_first = clean.fork("zero_first_relu")
zero_first.attach_hooks(tl.label(relu_labels[0]), tl.zero_ablate()).replay()

scale_second = clean.fork("scale_second_relu")
scale_second.attach_hooks(tl.label(relu_labels[1]), tl.scale(0.5)).replay()

logs = tl.bundle(
    {"clean": clean, "zero_first": zero_first, "scale_second": scale_second}, baseline="clean"
)
print(logs.names)

In [ ]:
output_norms = logs.metric(
    lambda member: float(torch.linalg.vector_norm(member.layer_list[-1].activation))
)
for name, value in output_norms.items():
    print(f"{name:16s} output norm {value:.4f}")

comparison = logs.compare_at(tl.label(relu_labels[0]))
print(f"pairwise comparison shape: {tuple(comparison.shape)}")

The bundle keeps branch names and common sites together. That matters once an analysis has many interventions, because metrics can be computed across members without manually aligning layer labels.
